<a href="https://colab.research.google.com/github/BenjaminOrtizO/CaseStudy/blob/main/Entrega1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#CASE OF STUDY: BEBIDAS MAIPO S.A.
##Parte 1: Calidad y Pronóstico.
###Cargar el repositorio


In [68]:
!git clone https://github.com/BenjaminOrtizO/CaseStudy.git

# Entrar al repositorio
%cd /content/CaseStudy


!git pull

fatal: destination path 'CaseStudy' already exists and is not an empty directory.
/content/CaseStudy
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
Unpacking objects: 100% (3/3), 11.17 KiB | 5.58 MiB/s, done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
From https://github.com/BenjaminOrtizO/CaseStudy
   5346d55..110545e  main       -> origin/main
Updating 5346d55..110545e
Fast-forward
 Entrega1.ipynb | 2239 +++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 1 file changed, 2221 insertions(+), 18 deletions(-)


###Cargar los datos


In [69]:
import pandas as pd
import numpy as np
import json

# Datos maestros
maestro_sku = pd.read_csv('Datos/maestro_sku.csv')
inventario_inicial = pd.read_csv('Datos/inventario_inicial.csv')

# Demanda y capacidad
ventas_semanal = pd.read_csv('Datos/demanda_semanal.csv')
capacidad_horizonte = pd.read_csv('Datos/capacidad_horizonte.csv')
compromisos_semana_focal = pd.read_csv('Datos/compromisos_semana_focal.csv')

# Calidad
calidad_atributos = pd.read_csv('Datos/calidad_atributos.csv')
calidad_llenado = pd.read_csv('Datos/calidad_llenado.csv')
especificaciones_calidad = pd.read_csv('Datos/especificaciones_calidad.csv')

# Configuración
setups_cip = pd.read_csv('Datos/setups_cip.csv')

# Metadatos
with open('Datos/metadatos.json', 'r', encoding='utf-8') as f:
    metadatos = json.load(f)

metadatos #equivalente a print en Jupyter

{'empresa': 'Bebidas del Maipo S.A.',
 'semilla': 20260923,
 'n_semanas_historia': 156,
 'fin_historia': '2026-09-21',
 'horizonte_planificacion': 13,
 'semana_focal_secuenciamiento': 12,
 'fecha_semana_focal': '2026-12-14',
 'n_sku': 9,
 'familias': ['CIT', 'FUN', 'NEC']}

##Preparación de los datos.


In [70]:
calidad_llenado.dtypes

,0
subgrupo,int64
fecha,object
semana_registro,int64
turno,object
linea,object
sku,object
formato_L,float64
hora,object
n_obs,int64
volumen_mL,float64


In [71]:
maestro_sku.dtypes

,0
sku,object
familia,object
nombre_familia,object
sabor,object
formato_L,float64
botellas_por_caja,int64
litros_por_caja,float64
linea_envasado,object
tasa_envasado_cajas_h,int64
costo_variable_caja_clp,int64


In [72]:
ventas_semanal.dtypes

,0
semana,int64
fecha_lunes,object
anio_iso,int64
semana_iso,int64
sku,object
ventas_cajas,float64
dias_sin_stock,float64
promocion,int64
precio_lista_clp,int64
precio_efectivo_clp,int64


In [73]:
calidad_atributos.dtypes

,0
fecha,object
semana_registro,int64
turno,object
linea,object
sku,object
n_inspeccionadas,int64
n_defectuosas,int64
causa_principal,object


In [74]:
especificaciones_calidad.dtypes

,0
formato_L,float64
nominal_mL,float64
LIE_mL,float64
LSE_mL,float64
tolerancia_oiml_mL,float64
limite_legal_inferior_mL,float64


In [75]:
calidad_llenado['fecha'] = pd.to_datetime(calidad_llenado['fecha'])
calidad_atributos['fecha'] = pd.to_datetime(calidad_atributos['fecha'])

calidad_atributos = calidad_atributos.set_index('fecha').sort_index()
calidad_llenado = calidad_llenado.set_index('fecha').sort_index()

In [76]:
calidad_llenado.info()
print('\n\n\n' , '='*300, '\n\n\n')
maestro_sku.info()
print('\n\n\n' , '='*300, '\n\n\n')
ventas_semanal.info()
print('\n\n\n' , '='*300, '\n\n\n')
calidad_atributos.info()
print('\n\n\n' , '='*300, '\n\n\n')
especificaciones_calidad.info()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4200 entries, 2026-07-20 to 2026-09-25
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   subgrupo         4200 non-null   int64  
 1   semana_registro  4200 non-null   int64  
 2   turno            4200 non-null   object 
 3   linea            4200 non-null   object 
 4   sku              4200 non-null   object 
 5   formato_L        4200 non-null   float64
 6   hora             4200 non-null   object 
 7   n_obs            4200 non-null   int64  
 8   volumen_mL       4200 non-null   float64
dtypes: float64(2), int64(3), object(4)
memory usage: 328.1+ KB






<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 13 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   sku                               9 non-null      object 
 1   famil

###Como se puede ver solo existen datos faltantes en el dataframe de ventas.

###En otras palabras, más adelante debemos tomar una decisión respecto a ello.

In [77]:
calidad_llenado.head(4200)

,subgrupo,semana_registro,turno,linea,sku,formato_L,hora,n_obs,volumen_mL
fecha,,,,,,,,,
2026-07-20,1,1,A (06-14),L1,FUN-B05,0.5,06:00,1,500.86
2026-07-20,30,1,B (14-22),L2,CIT-L30,3.0,16:00,4,3000.57
2026-07-20,30,1,B (14-22),L2,CIT-L30,3.0,16:00,3,3009.31
2026-07-20,30,1,B (14-22),L2,CIT-L30,3.0,16:00,2,3002.43
2026-07-20,30,1,B (14-22),L2,CIT-L30,3.0,16:00,1,3008.71
...,...,...,...,...,...,...,...,...,...
2026-09-25,833,10,B (14-22),L2,CIT-L30,3.0,14:00,5,3012.56
2026-09-25,834,10,B (14-22),L2,CIT-L30,3.0,16:00,1,3017.23
2026-09-25,834,10,B (14-22),L2,CIT-L30,3.0,16:00,2,3013.62


In [78]:
maestro_sku.head(4200)

,sku,familia,nombre_familia,sabor,formato_L,botellas_por_caja,litros_por_caja,linea_envasado,tasa_envasado_cajas_h,costo_variable_caja_clp,precio_lista_caja_clp,costo_inventario_caja_semana_clp,penalidad_atraso_caja_semana_clp
0,NEC-D05,NEC,Nectares con pulpa,Durazno,0.5,12,6.0,L1,180,3840,6620,95,1400
1,NEC-D15,NEC,Nectares con pulpa,Durazno,1.5,6,9.0,L2,210,5760,9930,95,1400
2,NEC-D30,NEC,Nectares con pulpa,Durazno,3.0,4,12.0,L2,130,7680,13240,95,1400
3,CIT-L05,CIT,Citricos gasificados,Limon,0.5,12,6.0,L1,180,2820,5320,95,1400
4,CIT-L15,CIT,Citricos gasificados,Limon,1.5,6,9.0,L2,210,4230,7980,95,1400
5,CIT-L30,CIT,Citricos gasificados,Limon,3.0,4,12.0,L2,130,5640,10640,95,1400
6,FUN-B05,FUN,Funcionales,Berries,0.5,12,6.0,L1,180,4860,9920,95,1400
7,FUN-B15,FUN,Funcionales,Berries,1.5,6,9.0,L2,210,7290,14880,95,1400
8,FUN-B30,FUN,Funcionales,Berries,3.0,4,12.0,L2,130,9720,19840,95,1400


In [79]:
ventas_semanal.head(4200)

,semana,fecha_lunes,anio_iso,semana_iso,sku,ventas_cajas,dias_sin_stock,promocion,precio_lista_clp,precio_efectivo_clp
0,1,2023-10-02,2023,40,NEC-D05,1209.0,0.0,0,6620,6620
1,2,2023-10-09,2023,41,NEC-D05,1095.0,0.0,0,6620,6620
2,3,2023-10-16,2023,42,NEC-D05,1345.0,0.0,0,6620,6620
3,4,2023-10-23,2023,43,NEC-D05,1888.0,0.0,1,6620,4819
4,5,2023-10-30,2023,44,NEC-D05,991.0,0.0,0,6620,6620
...,...,...,...,...,...,...,...,...,...,...
1399,152,2026-08-24,2026,35,FUN-B30,0.0,0.0,1,19840,14445
1400,153,2026-08-31,2026,36,FUN-B30,0.0,0.0,0,19840,19840
1401,154,2026-09-07,2026,37,FUN-B30,0.0,0.0,0,19840,19840
1402,155,2026-09-14,2026,38,FUN-B30,324.0,3.0,0,19840,19840


In [80]:
calidad_atributos.head(4200)

,semana_registro,turno,linea,sku,n_inspeccionadas,n_defectuosas,causa_principal
fecha,,,,,,,
2026-07-20,1,A (06-14),L1,FUN-B05,238,1,Envase deformado
2026-07-20,1,B (14-22),L1,CIT-L05,182,3,Etiqueta
2026-07-20,1,A (06-14),L2,CIT-L30,260,11,Etiqueta
2026-07-20,1,B (14-22),L2,CIT-L30,235,8,Cuerpo extranio
2026-07-20,1,C (22-06),L2,NEC-D15,215,2,Sello/tapa
...,...,...,...,...,...,...,...
2026-09-24,10,A (06-14),L2,CIT-L30,181,4,Nivel de llenado
2026-09-24,10,C (22-06),L2,NEC-D15,213,6,Etiqueta
2026-09-25,10,B (14-22),L2,CIT-L30,244,6,Sello/tapa


In [81]:
especificaciones_calidad.head(4200)

,formato_L,nominal_mL,LIE_mL,LSE_mL,tolerancia_oiml_mL,limite_legal_inferior_mL
0,0.5,500.0,492.0,508.0,15.0,485.0
1,1.5,1500.0,1485.0,1515.0,22.5,1477.5
2,3.0,3000.0,2975.0,3025.0,45.0,2955.0


##Los datos no permiten separar dos fuentes de variación, pues no se pueden identificar operadores ni instrumentos. Además, tampoco se ve que existan repeticiones.

##Por lo tanto, se procederá a establecer los límites.

#ETAPA A

##DATOS A UTILIZAR
###-calidad_llenado
###-calidad_atributos
###-especificaciones_calidad
###-maestro_sku
###-ventas_semanal


##Fase 1
###Establecer Límites.
####Previo a establecer los límites se debe saber si el instrumento está midiendo de manera apropiada. Para ello se debe realizar un análisis de MSA.

####Sin embargo, los datos no permitieron separar las fuentes de variación necesarias para el análisis. Así, se procede a establecer los límites.

####En el presente caso los subgrupos están establecidos con un n = 5 botellas, esto tiene sentido en el contexto del problema pues se está midiendo el volumen de llenado de las botellas.

In [83]:
#Agrupación base
calidad_llenado_sg = calidad_llenado.groupby(['linea', 'formato_L', 'subgrupo', 'semana_registro', 'turno', 'fecha', 'hora'])

#calcular media x y rango de los subgrupos
x_mean_sg = calidad_llenado_sg['volumen_mL'].mean()
rango_sg = calidad_llenado_sg['volumen_mL'].apply(lambda x: x.max() - x.min())

#Combinar en un DataFrame
calidad_llenado_sg = pd.DataFrame({'x_mean_sg': x_mean_sg,'rango_sg': rango_sg}).reset_index().sort_values(by=['linea','formato_L','fecha','hora'])
calidad_llenado_sg = calidad_llenado_sg.set_index('fecha').sort_index()

#Definir las Fases I y II, se suele preferir entre 20 y 25 subgrupos.
#Esto suele ser ideal porque permite evitar que se ensanchen los límites
#por culpa  de un valor atípico.






In [84]:
calidad_llenado_sg.head(840)

,linea,formato_L,subgrupo,semana_registro,turno,hora,x_mean_sg,rango_sg
fecha,,,,,,,,
2026-07-20,L1,0.5,1,1,A (06-14),06:00,501.560,4.01
2026-07-20,L2,1.5,36,1,C (22-06),04:00,1507.354,17.38
2026-07-20,L2,1.5,33,1,C (22-06),22:00,1505.642,25.95
2026-07-20,L2,3.0,25,1,A (06-14),06:00,3007.640,22.63
2026-07-20,L2,3.0,26,1,A (06-14),08:00,3006.826,16.15
...,...,...,...,...,...,...,...,...
2026-09-25,L2,3.0,835,10,B (14-22),18:00,3008.394,23.89
2026-09-25,L2,1.5,838,10,C (22-06),00:00,1509.408,10.09
2026-09-25,L2,1.5,839,10,C (22-06),02:00,1512.946,9.14
